# Pressure Visualization
Visualize upstream and downstream pressure over time from `cold_flow_data.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("cold_flow_data.csv")
df.columns = [str(c).strip() for c in df.columns]
df = df.rename(columns={"Unnamed: 2": "upstream_tag", "Unnamed: 4": "downstream_tag"})
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
df["Upstream Pressure"] = pd.to_numeric(df["Upstream Pressure"], errors="coerce")
df["Downstream Pressure"] = pd.to_numeric(df["Downstream Pressure"], errors="coerce")
df = df[df["START"].eq("DATA")].dropna(subset=["Time", "Upstream Pressure", "Downstream Pressure"])
df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure")
plt.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure")
plt.xlabel("Time")
plt.ylabel("Pressure")
plt.title("Upstream vs Downstream Pressure")
plt.legend()
plt.tight_layout()
plt.show()

## Savitzky-Golay Filter

Simulate a live savgol filter by taking the point in the middle of the filter window to avoid problematic behavior at the end of the curve if taking the last sample.

In [ ]:
from scipy.signal import savgol_filter

# Plot with multiple window lengths stacked vertically
window_lengths = [31, 51, 71]
polyorder = 3

if len(df) < max(window_lengths):
    raise ValueError("Not enough data points for the largest Savitzky-Golay window.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, window_length in zip(axes, window_lengths):
    if window_length % 2 == 0:
        window_length -= 1
    if window_length < 5:
        raise ValueError("Window length is too small for Savitzky-Golay filtering.")

    half_window = window_length // 2

    upstream_filtered = savgol_filter(df["Upstream Pressure"], window_length, polyorder)
    downstream_filtered = savgol_filter(df["Downstream Pressure"], window_length, polyorder)

    # Shift forward by half_window to simulate live reporting at the window center
    shifted_upstream = pd.Series(upstream_filtered).shift(half_window)
    shifted_downstream = pd.Series(downstream_filtered).shift(half_window)

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], shifted_upstream, label=f"Upstream (Savgol Live, {window_length})")
    ax.plot(df["Time"], shifted_downstream, label=f"Downstream (Savgol Live, {window_length})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Savgol Live Window Length {window_length}")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()

In [ ]:
ema_spans = [31, 51, 71]
if min(ema_spans) < 2:
    raise ValueError("EMA span must be >= 2.")

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 12), sharex=True)

for ax, ema_span in zip(axes, ema_spans):
    upstream_ema = df["Upstream Pressure"].ewm(span=ema_span, adjust=False).mean()
    downstream_ema = df["Downstream Pressure"].ewm(span=ema_span, adjust=False).mean()

    ax.plot(df["Time"], df["Upstream Pressure"], label="Upstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], df["Downstream Pressure"], label="Downstream Pressure (Raw)", alpha=0.4)
    ax.plot(df["Time"], upstream_ema, label=f"Upstream EMA (span={ema_span})")
    ax.plot(df["Time"], downstream_ema, label=f"Downstream EMA (span={ema_span})")
    ax.set_ylabel("Pressure")
    ax.set_title(f"Upstream vs Downstream Pressure (EMA span={ema_span})")
    ax.legend()

axes[-1].set_xlabel("Time")
fig.tight_layout()
plt.show()